# Define Operations Role Categories
Tags each occupation with one of the four analysis categories and produces a focused dataframe for downstream analysis.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED = Path('../data/processed')

master = pd.read_parquet(PROCESSED / 'master.parquet')
master['soc_7'] = master['soc_code'].str[:7]
print(f"Master: {len(master):,} occupations")

Master: 1,016 occupations


## 1. Category → SOC code mapping

In [2]:
CATEGORIES = {
    'Claims Specialists & Telephone Claims Reps': [
        '13-1031',    # Claims Adjusters, Examiners, and Investigators
        '43-9041',    # Insurance Claims and Policy Processing Clerks
        '13-2099.04', # Fraud Examiners, Investigators and Analysts
        '13-2053',    # Insurance Underwriters
    ],
    'Marketing & Communications': [
        '11-2021',    # Marketing Managers
        '11-2011',    # Advertising and Promotions Managers
        '11-2032',    # Public Relations Managers
        '13-1161',    # Market Research Analysts and Marketing Specialists
        '13-1161.01', # Search Marketing Strategists
        '27-3031',    # Public Relations Specialists
        '27-3099',    # Media and Communication Workers, All Other
        '41-3011',    # Advertising Sales Agents
    ],
    'Customer Service Representatives': [
        '43-4051',    # Customer Service Representatives
        '43-4171',    # Receptionists and Information Clerks
        '43-4151',    # Order Clerks
    ],
    'Physical Damage Adjusters': [
        '13-1032',    # Insurance Appraisers, Auto Damage
        '13-2022',    # Appraisers of Personal and Business Property
        '13-2023',    # Appraisers and Assessors of Real Estate
    ],
}

# Build reverse lookup: soc_code -> category
soc_to_category = {}
for cat, codes in CATEGORIES.items():
    for code in codes:
        soc_to_category[code] = cat

print(f"{sum(len(v) for v in CATEGORIES.values())} SOC codes across {len(CATEGORIES)} categories")

18 SOC codes across 4 categories


## 2. Tag and filter master

In [3]:
# Match on both full 10-digit code and 7-char prefix
def assign_category(row):
    if row['soc_code'] in soc_to_category:
        return soc_to_category[row['soc_code']]
    if row['soc_7'] in soc_to_category:
        return soc_to_category[row['soc_7']]
    return None

master['category'] = master.apply(assign_category, axis=1)
ops = master[master['category'].notna()].copy()

print(f"\n{len(ops)} occupations matched across categories:")
print(ops.groupby('category').size().rename('count').to_string())


18 occupations matched across categories:
category
Claims Specialists & Telephone Claims Reps    4
Customer Service Representatives              3
Marketing & Communications                    8
Physical Damage Adjusters                     3


## 3. Category summary table

In [4]:
summary = ops.groupby('category').agg(
    n_roles           = ('title', 'count'),
    total_employment  = ('employment', 'sum'),
    median_wage       = ('annual_median_wage', 'median'),
    avg_10yr_growth   = ('emp_change_pct', 'mean'),
    total_openings    = ('annual_openings', 'sum'),
    avg_automation    = ('automation_risk', 'mean'),
).round(2)

summary['total_employment'] = summary['total_employment'].map('{:,.0f}'.format)
summary['median_wage']      = summary['median_wage'].map('${:,.0f}'.format)
summary['total_openings']   = summary['total_openings'].map('{:,.0f}'.format)
summary['avg_10yr_growth']  = summary['avg_10yr_growth'].map('{:+.1f}%'.format)

print("Category Overview:")
summary

Category Overview:


,n_roles,total_employment,median_wage,avg_10yr_growth,total_openings,avg_automation
category,,,,,,
Claims Specialists & Telephone Claims Reps,4,"776,040","$79,550",-2.1%,60,0.70
Customer Service Representatives,3,"3,581,130","$44,770",-7.6%,478,0.62
Marketing & Communications,8,"2,685,390","$78,760",+3.0%,257,0.38
Physical Damage Adjusters,3,"11,560","$78,240",-8.2%,0,0.69


## 4. Role-level detail

In [5]:
detail_cols = [
    'category', 'soc_code', 'title',
    'employment', 'annual_median_wage',
    'emp_change_pct', 'annual_openings',
    'automation_risk', 'job_zone',
    'entry_education', 'on_job_training',
]

detail = ops[detail_cols].sort_values(['category', 'automation_risk'], ascending=[True, False])
detail = detail.reset_index(drop=True)

pd.set_option('display.max_rows', 30)
pd.set_option('display.max_colwidth', 50)
detail[['category','title','employment','annual_median_wage','emp_change_pct','automation_risk']]

,category,title,employment,annual_median_wage,emp_change_pct,automation_risk
0,Claims Specialists & Telephone Claims Reps,Insurance Claims and Policy Processing Clerks,214260.0,49230.0,-3.7,0.7932
1,Claims Specialists & Telephone Claims Reps,"Claims Adjusters, Examiners, and Investigators",324230.0,78000.0,-5.1,0.7044
2,Claims Specialists & Telephone Claims Reps,Insurance Underwriters,105420.0,81370.0,-2.6,0.6764
3,Claims Specialists & Telephone Claims Reps,"Fraud Examiners, Investigators and Analysts",132130.0,81100.0,3.1,0.6176
4,Customer Service Representatives,Order Clerks,75200.0,46170.0,-17.2,0.7038
5,Customer Service Representatives,Customer Service Representatives,2595750.0,44770.0,-5.5,0.5961
6,Customer Service Representatives,Receptionists and Information Clerks,910180.0,38010.0,0.0,0.5544
7,Marketing & Communications,Market Research Analysts and Marketing Special...,899580.0,78760.0,6.7,0.5369
8,Marketing & Communications,Public Relations Managers,74850.0,146910.0,5.0,0.5270
9,Marketing & Communications,"Media and Communication Workers, All Other",19590.0,73620.0,2.7,0.5270


## 5. Save

In [6]:
ops.to_parquet(PROCESSED / 'ops_roles.parquet', index=False)
detail.to_csv(PROCESSED / 'ops_roles_detail.csv', index=False)
print("Saved:")
print("  data/processed/ops_roles.parquet    — full dataset with all O*NET columns")
print("  data/processed/ops_roles_detail.csv — key metrics table")

Saved:
  data/processed/ops_roles.parquet    — full dataset with all O*NET columns
  data/processed/ops_roles_detail.csv — key metrics table
